# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset by schema URL
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets are organized into record sets, each potentially corresponding to a distinct resource or file containing tabular data. Fields correspond to columns in these record sets and are all referenced by their `@id`.

In [ ]:
# List all record sets, their @id and fields
record_sets = list(dataset.record_sets.keys())
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"RecordSet @id: {rs_id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field_id, field in record_set.fields.items():
        print(f"    Field @id: {field_id}")
        print(f"      Name: {field.name}")
        print(f"      Description: {field.description}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s as explored above.

Below, we'll load each record set and display their columns and a data sample.

In [ ]:
# Extract data from each record set by @id and display basic info
import warnings
warnings.filterwarnings('ignore')

dataframes = {}
for record_set_id in dataset.record_sets.keys():
    print(f"Loading records from RecordSet {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print("  Columns:", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"  Warning: Could not load records for {record_set_id}: {str(e)}")
print("Loaded DataFrames for:", list(dataframes.keys()))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping.

All references use `@id`. We'll select *one* record set and field for a typical numeric EDA, using IDs discovered above.

In [ ]:
# Example: Use the first available record set and a numeric field (if present) for EDA
if dataframes:
    # Select first record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Selected RecordSet @id: {record_set_id}")
    
    # Try to find a numeric field in columns
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        # If none are numeric, attempt to convert columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except Exception:
                continue
    if numeric_field:
        print(f"Numeric field selected: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notna(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Try grouping by another field (choose first object type column)
        group_field = None
        for col in df.columns:
            if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group-by field found.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All field/column references use their `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field} (@id) in RecordSet {record_set_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Visualize group means if grouping occurred
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f'Mean {numeric_field} by {group_field} (@id)')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've explored the Mass Spectrometry dataset using the `mlcroissant` library.

- We loaded metadata and listed all record sets and their fields by their Croissant `@id`.
- We extracted records from the main record set and performed basic EDA: filtering, normalizing, and grouping data using field `@id` only.
- We visualized core numeric fields for distribution and for group-level statistics.

This structure can be extended for advanced analysis or integrated into data processing pipelines as needed.